# Hyperparameter Tuning — Extra Trees on AP2D_Count

**Best model from screening**: Extra Trees on AtomPairs2D Count fingerprint (Random split)
- Regression baseline: R² = 0.697, RMSE = 0.713
- Classification baseline: BalAcc = 0.688, MCC = 0.540

**Method**: RandomizedSearchCV, 100 iterations, 5-fold CV


In [1]:
import os

if not os.path.exists("data"):
    !wget -q "https://github.com/dom-castaneda/qsar-aromatase/raw/master/data.zip" -O data.zip
    !unzip -qo data.zip
    print("Data extracted.")
else:
    print("Data present.")
print("Setup complete.")


Data extracted.
Setup complete.


In [2]:
import time, json
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, ExtraTreesClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                             balanced_accuracy_score, f1_score, matthews_corrcoef)
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42
N_ITER = 100
N_FOLDS = 5
BASE = "data"

# Load data
df_full = pd.read_csv(f"{BASE}/processed/aromatase_bioactivity_clean.csv")
mask = (df_full["standard_relation"] == "=") & df_full["pchembl_value"].notna()
df = df_full[mask].reset_index(drop=True)

# Load AP2D_Count fingerprint
fp_full = pd.read_csv(f"{BASE}/fingerprints_filtered/fingerprints_atompairs2d_count.csv")
fp = fp_full[mask.values].reset_index(drop=True)
fp_cols = [c for c in fp.columns if c != "molecule_chembl_id"]
X_all = np.nan_to_num(fp[fp_cols].values.astype(np.float32), nan=0.0)

# Regression target
y_reg = df["pchembl_value"].values

# Classification target
def classify(val):
    if val > 7: return "active"
    elif val < 6: return "inactive"
    else: return "intermediate"

le = LabelEncoder()
y_cls = le.fit_transform(df["pchembl_value"].apply(classify).values)
label_names = list(le.classes_)

# Train/test split
train_ids = set(pd.read_csv(f"{BASE}/splits/random_train.csv")["molecule_chembl_id"])
test_ids = set(pd.read_csv(f"{BASE}/splits/random_test.csv")["molecule_chembl_id"])
train_mask = df["molecule_chembl_id"].isin(train_ids).values
test_mask = df["molecule_chembl_id"].isin(test_ids).values

X_train, X_test = X_all[train_mask], X_all[test_mask]
y_train_reg, y_test_reg = y_reg[train_mask], y_reg[test_mask]
y_train_cls, y_test_cls = y_cls[train_mask], y_cls[test_mask]

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]} | Features: {X_train.shape[1]}")
print(f"Regression target: pchembl_value (continuous)")
print(f"Classification target: {dict(zip(label_names, le.transform(label_names)))}")


Train: 2732 | Test: 758 | Features: 455
Regression target: pchembl_value (continuous)
Classification target: {'active': np.int64(0), 'inactive': np.int64(1), 'intermediate': np.int64(2)}


In [3]:
# Hyperparameter search space
param_dist = {
    "n_estimators": [200, 500, 800, 1000],
    "max_depth": [None, 20, 30, 50, 70],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 6],
    "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7, None],
}

total_combos = 1
for v in param_dist.values():
    total_combos *= len(v)
print(f"Search space: {total_combos} total combinations")
print(f"Sampling {N_ITER} random configurations x {N_FOLDS}-fold CV = {N_ITER * N_FOLDS} fits")


Search space: 1920 total combinations
Sampling 100 random configurations x 5-fold CV = 500 fits


In [4]:
print("=" * 60)
print("REGRESSION TUNING: Extra Trees on AP2D_Count")
print("=" * 60)

t0 = time.time()
reg_search = RandomizedSearchCV(
    ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=N_ITER,
    cv=N_FOLDS,
    scoring="r2",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=2,
)
reg_search.fit(X_train, y_train_reg)
reg_time = time.time() - t0

print(f"\nCompleted in {reg_time/60:.1f} min")
print(f"Best CV R\u00b2: {reg_search.best_score_:.4f}")
print(f"Best params: {reg_search.best_params_}")

# Test evaluation
best_reg = reg_search.best_estimator_
y_pred_train = best_reg.predict(X_train)
y_pred_test = best_reg.predict(X_test)

r2_train = r2_score(y_train_reg, y_pred_train)
r2_test = r2_score(y_test_reg, y_pred_test)
rmse_train = np.sqrt(mean_squared_error(y_train_reg, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test_reg, y_pred_test))
mae_train = mean_absolute_error(y_train_reg, y_pred_train)
mae_test = mean_absolute_error(y_test_reg, y_pred_test)

print(f"\nTest Performance (tuned):")
print(f"  R\u00b2:   train={r2_train:.4f} | test={r2_test:.4f}")
print(f"  RMSE: train={rmse_train:.4f} | test={rmse_test:.4f}")
print(f"  MAE:  train={mae_train:.4f} | test={mae_test:.4f}")
print(f"\nBaseline (default params): R\u00b2=0.6967, RMSE=0.7128")
print(f"Improvement: R\u00b2 {r2_test - 0.6967:+.4f}, RMSE {rmse_test - 0.7128:+.4f}")


REGRESSION TUNING: Extra Trees on AP2D_Count
Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END max_depth=30, max_features=0.5, min_samples_leaf=4, min_samples_split=5, n_estimators=800; total time=   5.1s
[CV] END max_depth=30, max_features=0.5, min_samples_leaf=4, min_samples_split=5, n_estimators=800; total time=   4.9s
[CV] END max_depth=30, max_features=0.5, min_samples_leaf=4, min_samples_split=5, n_estimators=800; total time=   4.4s
[CV] END max_depth=30, max_features=0.5, min_samples_leaf=4, min_samples_split=5, n_estimators=800; total time=   4.3s
[CV] END max_depth=30, max_features=0.5, min_samples_leaf=4, min_samples_split=5, n_estimators=800; total time=   4.7s
[CV] END max_depth=70, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=500; total time=   0.7s
[CV] END max_depth=70, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=500; total time=   0.7s
[CV] END max_depth=70, max_features=log2, min_samples_lea

In [5]:
print("=" * 60)
print("CLASSIFICATION TUNING: Extra Trees on AP2D_Count")
print("=" * 60)

t0 = time.time()
cls_search = RandomizedSearchCV(
    ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"),
    param_distributions=param_dist,
    n_iter=N_ITER,
    cv=N_FOLDS,
    scoring="balanced_accuracy",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=2,
)
cls_search.fit(X_train, y_train_cls)
cls_time = time.time() - t0

print(f"\nCompleted in {cls_time/60:.1f} min")
print(f"Best CV BalAcc: {cls_search.best_score_:.4f}")
print(f"Best params: {cls_search.best_params_}")

# Test evaluation
best_cls = cls_search.best_estimator_
y_pred_train_cls = best_cls.predict(X_train)
y_pred_test_cls = best_cls.predict(X_test)

balacc_train = balanced_accuracy_score(y_train_cls, y_pred_train_cls)
balacc_test = balanced_accuracy_score(y_test_cls, y_pred_test_cls)
mcc_test = matthews_corrcoef(y_test_cls, y_pred_test_cls)
f1_test = f1_score(y_test_cls, y_pred_test_cls, average="weighted")

print(f"\nTest Performance (tuned):")
print(f"  BalAcc: train={balacc_train:.4f} | test={balacc_test:.4f}")
print(f"  MCC:   {mcc_test:.4f}")
print(f"  F1:    {f1_test:.4f}")
print(f"\nBaseline (default params): BalAcc=0.6883, MCC=0.5401")
print(f"Improvement: BalAcc {balacc_test - 0.6883:+.4f}, MCC {mcc_test - 0.5401:+.4f}")


CLASSIFICATION TUNING: Extra Trees on AP2D_Count
Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END max_depth=30, max_features=0.5, min_samples_leaf=4, min_samples_split=5, n_estimators=800; total time=   4.3s
[CV] END max_depth=30, max_features=0.5, min_samples_leaf=4, min_samples_split=5, n_estimators=800; total time=   4.5s
[CV] END max_depth=30, max_features=0.5, min_samples_leaf=4, min_samples_split=5, n_estimators=800; total time=   4.4s
[CV] END max_depth=30, max_features=0.5, min_samples_leaf=4, min_samples_split=5, n_estimators=800; total time=   4.2s
[CV] END max_depth=30, max_features=0.5, min_samples_leaf=4, min_samples_split=5, n_estimators=800; total time=   4.4s
[CV] END max_depth=70, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=500; total time=   1.0s
[CV] END max_depth=70, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=500; total time=   1.0s
[CV] END max_depth=70, max_features=log2, min_samples

In [6]:
# Summary
print("=" * 60)
print("TUNING SUMMARY")
print("=" * 60)
print(f"\nRegression (Extra Trees on AP2D_Count, Random split):")
print(f"  Default:  R\u00b2={0.6967:.4f}, RMSE={0.7128:.4f}")
print(f"  Tuned:    R\u00b2={r2_test:.4f}, RMSE={rmse_test:.4f}")
print(f"  Best params: {reg_search.best_params_}")
print(f"\nClassification (Extra Trees on AP2D_Count, Random split):")
print(f"  Default:  BalAcc={0.6883:.4f}, MCC={0.5401:.4f}")
print(f"  Tuned:    BalAcc={balacc_test:.4f}, MCC={mcc_test:.4f}")
print(f"  Best params: {cls_search.best_params_}")

# Save results
results = {
    "regression": {
        "best_params": {k: (int(v) if isinstance(v, (int, np.integer)) else
                           float(v) if isinstance(v, (float, np.floating)) else v)
                       for k, v in reg_search.best_params_.items()},
        "best_cv_r2": float(reg_search.best_score_),
        "test_r2": float(r2_test),
        "test_rmse": float(rmse_test),
        "test_mae": float(mae_test),
        "train_r2": float(r2_train),
        "baseline_r2": 0.6967,
        "tuning_time_min": round(reg_time / 60, 1),
    },
    "classification": {
        "best_params": {k: (int(v) if isinstance(v, (int, np.integer)) else
                           float(v) if isinstance(v, (float, np.floating)) else v)
                       for k, v in cls_search.best_params_.items()},
        "best_cv_balacc": float(cls_search.best_score_),
        "test_balacc": float(balacc_test),
        "test_mcc": float(mcc_test),
        "test_f1": float(f1_test),
        "train_balacc": float(balacc_train),
        "baseline_balacc": 0.6883,
        "tuning_time_min": round(cls_time / 60, 1),
    },
    "fingerprint": "AP2D_Count",
    "split": "Random",
    "n_iter": N_ITER,
    "n_folds": N_FOLDS,
}

with open("tuning_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved: tuning_results.json")


TUNING SUMMARY

Regression (Extra Trees on AP2D_Count, Random split):
  Default:  R²=0.6967, RMSE=0.7128
  Tuned:    R²=0.6762, RMSE=0.7364
  Best params: {'n_estimators': 1000, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 0.3, 'max_depth': 20}

Classification (Extra Trees on AP2D_Count, Random split):
  Default:  BalAcc=0.6883, MCC=0.5401
  Tuned:    BalAcc=0.6896, MCC=0.5433
  Best params: {'n_estimators': 800, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.3, 'max_depth': 20}

Saved: tuning_results.json


In [7]:
from google.colab import files
files.download("tuning_results.json")
print("Done.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done.
